# VGG16 pipeline: cell classification on PCMMD, patients folder only (Google Colab)

Data used: **only** `data/detection/patients/` (patient 01 to patient 10, each with `images/` and `labels/`). The `segmentation/` folders and `detection/train/` are deliberately **not** used.
Split: **60% train / 10% validation / 30% test**, done **by patient** (6 / 1 / 3 patients). Set **Runtime > T4 GPU** first.

| Step | What happens |
|---|---|
| 0 | GPU check, imports, seed |
| 1 | Mount Drive, point at the `patients` folder |
| 2 | Copy only that folder to local disk, check each patient |
| 3 | Class names (yaml) and diagnosis.csv |
| 4 | Cut every labelled cell out of every photo |
| 5 | Look at crops, choose the positive class |
| 6 | Split 60 / 10 / 30 |
| 7 | Class weights |
| 8 | Data pipeline + augmentation |
| 9 | Build VGG16 |
| 10 | Phase 1: train the new head |
| 11 | Phase 2: fine-tune |
| 12 | Learning curves |
| 13 | Test evaluation (overall and per patient) |
| 14 | Look at mistakes |
| 15 | Save to Drive |

## Step 0: GPU, imports, seed
**Why:** training needs a GPU to be fast. Importing everything once keeps later cells short. A fixed seed makes the split and training repeatable.

In [ ]:
import os, glob, shutil, random, time, json, re, itertools
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd, cv2, yaml
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve,
                             accuracy_score, precision_score, recall_score, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## Step 1: Mount Drive and point at the `patients` folder
**Why:** Colab can only read your files after Drive is mounted. **The important fix:** the path now goes all the way down to `.../data/detection/patients`, not just `PlasmaCells`. Pointing at the top folder made the old notebook walk into `segmentation/` and `detection/train/` too. The `assert` makes a typo fail here, not later.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/PlasmaCells/PCMMD Plasma Cells for Multiple Myeloma Diagnosis/data/detection"
PATIENTS_DIR = f"{BASE}/patients"
assert os.path.isdir(PATIENTS_DIR), f"not found: {PATIENTS_DIR}"
print(sorted(os.listdir(PATIENTS_DIR)))

## Step 2: Copy only `patients/` to local disk and check each patient
**Why copy:** reading thousands of small files from Drive is slow; Colab's local disk is fast. **Why only this folder:** so nothing else can sneak in. **Why check:** each patient should have `images/` and `labels/`, with one `.txt` per photo sharing its file name. We count them so a missing or mismatched folder shows up now.

In [ ]:
LOCAL = "/content/patients"
if not os.path.isdir(LOCAL):
    shutil.copytree(PATIENTS_DIR, LOCAL)

IMG_EXT = (".bmp", ".jpg", ".jpeg", ".png", ".tif", ".tiff")
rows = []
for pdir in sorted(p for p in glob.glob(f"{LOCAL}/*") if os.path.isdir(p)):
    ims = [f for f in os.listdir(f"{pdir}/images") if f.lower().endswith(IMG_EXT)] if os.path.isdir(f"{pdir}/images") else []
    lbs = [f for f in os.listdir(f"{pdir}/labels") if f.endswith(".txt")] if os.path.isdir(f"{pdir}/labels") else []
    both = {Path(f).stem for f in ims} & {Path(f).stem for f in lbs}
    rows.append((os.path.basename(pdir), len(ims), len(lbs), len(both)))
layout = pd.DataFrame(rows, columns=["patient", "images", "label_files", "matched"])
display(layout)
print("patients:", len(layout), "| photos:", layout.images.sum(), "| photos with labels:", layout.matched.sum())

## Step 3: Class names and diagnosis.csv
**Class names:** YOLO label files only hold numbers (`0`, `1`). The meaning is in the `.yaml` next to them, so we print it instead of guessing. **Read the `names:` part.**

**diagnosis.csv:** per **patient**, not per cell, so it can't label individual cells. We display it so you can describe your patients and check the split.

In [ ]:
y = f"{BASE}/mieloma_final.yaml"
print(open(y).read()[:800] if os.path.exists(y) else glob.glob(f"{BASE}/*.yaml"))

In [ ]:
diag_path = f"{LOCAL}/diagnosis.csv"
if os.path.exists(diag_path):
    diag = pd.read_csv(diag_path); print(diag.shape); display(diag)
else:
    print("diagnosis.csv not found in", LOCAL)

## Step 4: Cut every labelled cell out of every photo
**Why:** the dataset is for *detection* (whole photos with boxes), but a classifier needs one image per cell. For each line `class x_center y_center width height` (relative to the photo) we convert to pixels, add a 10% border so cell edges aren't clipped, skip boxes under 24 px (too small to classify), and save the crop. Each crop remembers its **patient** and **photo**, which the split needs.

In [ ]:
CROP_DIR, PAD, MIN_SIDE = Path("/content/crops"), 0.10, 24
rows, skipped_small, skipped_bad = [], 0, 0
for pdir in sorted(p for p in glob.glob(f"{LOCAL}/*") if os.path.isdir(p)):
    patient = os.path.basename(pdir)
    img_by_stem = {Path(f).stem: f"{pdir}/images/{f}" for f in os.listdir(f"{pdir}/images") if f.lower().endswith(IMG_EXT)}
    for lf in sorted(os.listdir(f"{pdir}/labels")):
        stem = Path(lf).stem
        if not lf.endswith(".txt") or stem not in img_by_stem: continue
        im = cv2.imread(img_by_stem[stem])
        if im is None: skipped_bad += 1; continue
        H, W = im.shape[:2]
        for i, line in enumerate(open(f"{pdir}/labels/{lf}").read().strip().splitlines()):
            p = line.split()
            if len(p) < 5: continue
            c = int(float(p[0])); xc, yc, w, h = map(float, p[1:5])
            bw, bh = w * W, h * H
            x1, x2 = int(max(0, xc*W - bw/2 - PAD*bw)), int(min(W, xc*W + bw/2 + PAD*bw))
            y1, y2 = int(max(0, yc*H - bh/2 - PAD*bh)), int(min(H, yc*H + bh/2 + PAD*bh))
            if min(x2 - x1, y2 - y1) < MIN_SIDE: skipped_small += 1; continue
            out = CROP_DIR / patient / f"class_{c}" / f"{stem}_{i}.png"
            out.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(out), im[y1:y2, x1:x2])
            rows.append((str(out), str(c), patient, f"{patient}/{stem}"))
df = pd.DataFrame(rows, columns=["path", "class_name", "patient", "photo"])
print(f"{len(df)} crops | skipped too small: {skipped_small} | unreadable photos: {skipped_bad}")
display(pd.crosstab(df.patient, df.class_name, margins=True))

## Step 5: Look at the crops, then choose the positive class
**Why:** a number like `1` means nothing until you've seen it. 8 random crops per class id; combine with the yaml `names:` from Step 3.

In [ ]:
names = sorted(df.class_name.unique())
fig, axes = plt.subplots(len(names), 8, figsize=(20, 2.6 * len(names)), squeeze=False)
for r, n in enumerate(names):
    sub = df[df.class_name == n]; sample = sub.sample(min(8, len(sub)), random_state=0)
    for ax in axes[r]: ax.axis("off")
    for ax, p in zip(axes[r], sample.path): ax.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    axes[r][0].set_title(f"class id = {n}", loc="left")
plt.show()

**Choose the positive class.** As far as I could tell, this dataset's expert labels are *plasma cell vs other cell*, not malignant vs normal, so "positive = plasma cell" is a **proxy** for myeloma cells. Confirm which id is the plasma cell from the yaml `names:` and the pictures, then set it below. Everything that is not `POSITIVE_CLASS` becomes 0.

In [ ]:
POSITIVE_CLASS = "1"     # <- CHECK against the yaml names in Step 3 and the pictures above
POS_NAME, NEG_NAME = "positive", "negative"     # <- optionally rename, e.g. "plasma", "non-plasma"

df["label"] = (df.class_name == POSITIVE_CLASS).astype(int)
assert df.label.nunique() == 2, "Only one class present: check POSITIVE_CLASS"
print(df.label.value_counts().rename({1: POS_NAME, 0: NEG_NAME}))

# Fix this : Training: 75%, testing 15%, Validation 10%
## Step 6: Split 60% train / 10% validation / 30% test
- **train** (60%): the model learns from it. **validation** (10%): we watch it to pick the best epoch and stop early. **test** (30%): touched **once**, at the end.
- **`SPLIT_MODE = "patient"` (default, recommended):** whole patients go to one split. With 10 patients that is **6 train / 1 val / 3 test**. Cells of one patient share the slide, stain, lighting and camera, so mixing patients across splits lets the model partly memorise the patient and inflates the test score. This split asks the question that matters: *does it work on a new patient?*
  Percentages are by **cells**, so they are only *approximately* 60/10/30, because patients have different numbers of cells. The code tries all 840 ways to pick 1 val + 3 test patients and chooses the one whose sizes are closest to 10% / 30% and whose class balance is closest to the overall balance (both classes must be present in val and test). Actual percentages are printed.
- **`SPLIT_MODE = "photo"`:** exact 60/10/30 by cells, but whole *photos* stay together. Patients are then shared between splits, so expect optimistic scores. Use it only if you need the exact numbers, and say so in the report.

In [ ]:
# SPLIT_MODE = "patient"        # "patient" (recommended) or "photo"
# MANUAL_SPLIT = None           # patient mode only, e.g. {"val": ["patient 03"], "test": ["patient 01", "patient 05", "patient 08"]}
# N_VAL, N_TEST, TARGET_VAL, TARGET_TEST = 1, 3, 0.10, 0.30

# if SPLIT_MODE == "patient":
#     patients = sorted(df.patient.unique())
#     cnt = df.groupby("patient").label.agg(pos="sum", n="count")
#     N = cnt.n.sum(); overall = cnt.pos.sum() / N

#     def score(vp, tp, minc):
#         s = 0
#         for grp, target in ((vp, TARGET_VAL), (tp, TARGET_TEST)):
#             pos, n = cnt.loc[list(grp), "pos"].sum(), cnt.loc[list(grp), "n"].sum()
#             if pos < minc or n - pos < minc: return None
#             s += abs(pos / n - overall) + abs(n / N - target)
#         return s

#     if MANUAL_SPLIT:
#         val_p, test_p = MANUAL_SPLIT["val"], MANUAL_SPLIT["test"]
#     else:
#         best = None
#         for minc in (30, 20, 10, 5):                    # relax the minimum cells per class if needed
#             for vp in itertools.combinations(patients, N_VAL):
#                 rest = [p for p in patients if p not in vp]
#                 for tp in itertools.combinations(rest, N_TEST):
#                     sc = score(vp, tp, minc)
#                     if sc is not None and (best is None or sc < best[0]): best = (sc, vp, tp)
#             if best: print(f"(each of val/test has at least {minc} cells of each class)"); break
#         assert best, "No valid patient split: use MANUAL_SPLIT or SPLIT_MODE='photo'"
#         val_p, test_p = list(best[1]), list(best[2])
#     train_p = [p for p in patients if p not in val_p + test_p]
#     tr, val, test = (df[df.patient.isin(g)] for g in (train_p, val_p, test_p))
# else:
#     g1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
#     a, b = next(g1.split(df, df.label, df.photo)); trval, test = df.iloc[a], df.iloc[b]
#     g2 = GroupShuffleSplit(n_splits=1, test_size=0.10 / 0.70, random_state=SEED)
#     a, b = next(g2.split(trval, trval.label, trval.photo)); tr, val = trval.iloc[a], trval.iloc[b]

# tr, val, test = tr.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)
# train_p, val_p, test_p = (sorted(d.patient.unique()) for d in (tr, val, test))
# for n, d, g in [("train", tr, train_p), ("val", val, val_p), ("test", test, test_p)]:
#     print(f"{n:5s} {len(d):6d} cells ({len(d)/len(df):5.1%}) | {POS_NAME}={int(d.label.sum())} {NEG_NAME}={int((1-d.label).sum())} | {g}")

In [ ]:
from sklearn.model_selection import train_test_split


trval, test = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=SEED
)

#From the remaining 85 %, take ~11.8 % as validation → overall 10 % of the whole data
#    (0.10 / 0.85 ≈ 0.1176)
tr, val = train_test_split(
    trval,
    test_size=0.10 / 0.85,
    stratify=trval["label"],
    random_state=SEED
)

tr  = tr.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

train_p = sorted(tr.patient.unique())
val_p   = sorted(val.patient.unique())
test_p  = sorted(test.patient.unique())

print("=== POOLED (cell-level) split – patients are shared ===")
for name, d, g in [("train", tr, train_p), ("val", val, val_p), ("test", test, test_p)]:
    n_pos = int(d.label.sum())
    n_neg = len(d) - n_pos
    print(f"{name:5s} {len(d):6d} cells ({len(d)/len(df):5.1%}) | "
          f"{POS_NAME}={n_pos}  {NEG_NAME}={n_neg} | patients: {g}")

## Check for the Patient distribusion

In [ ]:
def group(p):
    # assumes folder names like "patient 01", "patient 1", "01", etc.
    num = int("".join(c for c in p if c.isdigit()))
    return "deceased (01-05)" if num <= 5 else "alive (06-10)"

df["clinical_group"] = df.patient.apply(group)
print(df.groupby(["clinical_group", "label"]).size().unstack(fill_value=0))
print("\nAfter split:")
for name, d in [("train", tr), ("val", val), ("test", test)]:
    print(name, d.groupby("clinical_group").size().to_dict())

## Step 7: Class weights
**Why:** if one class is much more common, the model can score well by always guessing it. Weights make errors on the rarer class cost more during training. Computed from the **training set only**, so nothing leaks from val/test.

In [ ]:
n0, n1 = (tr.label == 0).sum(), (tr.label == 1).sum()
CLASS_WEIGHT = {0: len(tr) / (2 * n0), 1: len(tr) / (2 * n1)}
print(CLASS_WEIGHT)

## Step 8: Data pipeline and augmentation
- **Resize to 224x224** (VGG16's pre-trained size). Small cell crops get enlarged; that's fine.
- **Batches of 32, prefetched** so the GPU doesn't wait for the disk.
- **Augmentation** (flips, rotation, small zoom), only while training: more variety from limited data. Cells have no 'up', so flips and rotations are safe. No colour changes, since stain colour may carry information.
- **`preprocess_input`** is VGG16's own pixel preprocessing (RGB to BGR, subtract the ImageNet channel means). It sits inside the model so train, val and test are treated identically.

In [ ]:
SIZE, BATCH = 224, 32
def make_ds(frame, train):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label.values.astype("float32")))
    if train: ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    def load(p, y):
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img = tf.image.resize(img, (SIZE, SIZE)); img.set_shape((SIZE, SIZE, 3))
        return img, y
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(tr, True), make_ds(val, False), make_ds(test, False)
augment = keras.Sequential([layers.RandomFlip("horizontal_and_vertical"),
                            layers.RandomRotation(0.25), layers.RandomZoom(0.1)], name="augment")

## Step 9: Build VGG16
**Why:** VGG16 was already trained on ImageNet, so it knows edges, textures and shapes; that helps a lot with limited medical data.
- Load it **without its original 1000-class top** (`include_top=False`) and **freeze** it for now.
- New head: global average pooling, Dense(256), dropout 0.5 (randomly switches off half the units so it doesn't memorise), and one **sigmoid** unit = probability the cell is positive.

In [ ]:
def build_vgg16():
    inp = keras.Input((SIZE, SIZE, 3))
    x = augment(inp)
    x = layers.Lambda(preprocess_input, name="vgg_preprocess")(x)
    base = VGG16(include_top=False, weights="imagenet", input_shape=(SIZE, SIZE, 3))
    base.trainable = False
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out, name="VGG16_classifier"), base

model, base = build_vgg16()
METRICS = [keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.AUC(name="auc"),
           keras.metrics.Precision(name="prec"), keras.metrics.Recall(name="rec")]
OUT = "/content/drive/MyDrive/VGG16_results"; os.makedirs(OUT, exist_ok=True)
def cbs():
    return [keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1)]
print("parameters:", f"{model.count_params()/1e6:.2f} M")

## Step 10: Phase 1, train only the new head
**Why first:** the new output layer starts random. If we trained everything now, its large random errors would flow back and damage the good pre-trained weights. So VGG16's convolution layers are frozen; only our new layers learn (learning rate 1e-3). **Early stopping** on validation AUC (patience 5) stops when validation stops improving and restores the best epoch.

In [ ]:
t0 = time.time()
model.compile(keras.optimizers.Adam(1e-3), "binary_crossentropy", metrics=METRICS)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=CLASS_WEIGHT, callbacks=cbs(), verbose=2)

## Step 11: Phase 2, fine-tune the last convolution block
**Why:** the head is now sensible, so we let VGG16's **`block5`** (its last 3 conv layers, the most task-specific) adapt to blood cells; earlier blocks stay frozen. Learning rate drops to 1e-5 because big steps would erase what it already knows. We must **recompile** after changing which layers are trainable, otherwise the change is ignored.

In [ ]:
base.trainable = True
for l in base.layers:
    l.trainable = l.name.startswith("block5")
print("trainable VGG layers:", [l.name for l in base.layers if l.trainable])

model.compile(keras.optimizers.Adam(1e-5), "binary_crossentropy", metrics=METRICS)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=CLASS_WEIGHT, callbacks=cbs(), verbose=2)
print(f"training time: {(time.time()-t0)/60:.1f} min")

## Step 12: Learning curves
**Why:** to see *how* it trained. Dashed line = start of fine-tuning. Training improving while validation stalls or worsens means overfitting. With only **1 validation patient**, expect the validation curve to be a bit jumpy.

In [ ]:
def cat(k): return h1.history[k] + h2.history[k]
fig, ax = plt.subplots(1, 2, figsize=(13, 4)); k1 = len(h1.history["loss"])
for a, k, t in [(ax[0], "loss", "Loss"), (ax[1], "auc", "AUC")]:
    a.plot(cat(k), label="train"); a.plot(cat("val_" + k), label="validation")
    a.axvline(k1 - 0.5, ls="--", c="grey"); a.set_title(t); a.set_xlabel("epoch"); a.legend()
plt.savefig(f"{OUT}/curves.png", bbox_inches="tight"); plt.show()

## Step 13: Test evaluation, the one time we touch the test set
**Why these metrics:** accuracy alone can hide failure on the rarer class. *Precision* = of cells called positive, how many were. *Recall* = of true positives, how many we found. *F1* combines them. *AUC* measures ranking quality independent of the 0.5 cut-off. The **per-patient table** shows whether it works on each test patient or only some.

In [ ]:
y_true = test.label.values
y_prob = model.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print(classification_report(y_true, y_pred, target_names=[NEG_NAME, POS_NAME], digits=4))

res = dict(split_mode=SPLIT_MODE, accuracy=accuracy_score(y_true, y_pred), precision=precision_score(y_true, y_pred, zero_division=0),
           recall=recall_score(y_true, y_pred), f1=f1_score(y_true, y_pred), auc=float(roc_auc_score(y_true, y_prob)),
           n_train=len(tr), n_val=len(val), n_test=len(test), train_patients=train_p, val_patients=val_p, test_patients=test_p)
print({k: round(v, 4) if isinstance(v, float) else v for k, v in res.items()})

test["prob"], test["pred"] = y_prob, y_pred
per = test.groupby("patient").apply(lambda d: pd.Series({
    "cells": len(d), "accuracy": accuracy_score(d.label, d.pred),
    "recall": recall_score(d.label, d.pred, zero_division=0),
    "precision": precision_score(d.label, d.pred, zero_division=0)}))
display(per.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues", ax=ax[0],
            xticklabels=[NEG_NAME, POS_NAME], yticklabels=[NEG_NAME, POS_NAME])
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true"); ax[0].set_title("Confusion matrix (test)")
fpr, tpr, _ = roc_curve(y_true, y_prob)
ax[1].plot(fpr, tpr, label=f"AUC = {res['auc']:.3f}"); ax[1].plot([0, 1], [0, 1], "k--")
ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate"); ax[1].set_title("ROC (test)"); ax[1].legend()
plt.savefig(f"{OUT}/test_eval.png", bbox_inches="tight"); plt.show()

## GradCAM&GRADCAM++


In [ ]:
# ==========================================
# NEW STEP: Grad-CAM and Grad-CAM++
# ==========================================
!pip install -q tf-keras-vis

In [ ]:


import numpy as np
from matplotlib import cm
from tf_keras_vis.gradcam import Gradcam, GradcamPlusPlus
from tf_keras_vis.utils.scores import BinaryScore

# 1. Prepare a few validation images
sample_val = val.sample(4, random_state=SEED)
img_arrays = []
for path in sample_val.path:
    img = tf.keras.utils.load_img(path, target_size=(SIZE, SIZE))
    img_arrays.append(tf.keras.utils.img_to_array(img))
X_val_sample = np.array(img_arrays)

# 2. Define the score function (we want to explain the 'positive' class prediction)
# Since it's binary classification with sigmoid, output is 1 value.
score = BinaryScore(1.0)

# 3. Replace the model's activation with linear for better gradient flow (Standard practice)
modifier_model = tf.keras.models.clone_model(model)
modifier_model.set_weights(model.get_weights())
modifier_model.layers[-1].activation = tf.keras.activations.linear

# 4. Initialize Explainers
gradcam = Gradcam(modifier_model, clone=False)
gradcam_plus = GradcamPlusPlus(modifier_model, clone=False)

# 5. Generate Heatmaps
cam_heatmaps = gradcam(score, X_val_sample, penultimate_layer=-1)
cam_plus_heatmaps = gradcam_plus(score, X_val_sample, penultimate_layer=-1)

# 6. Plotting
fig, axes = plt.subplots(len(sample_val), 3, figsize=(12, 4 * len(sample_val)))
for i, (idx, row) in enumerate(sample_val.iterrows()):
    img = np.uint8(X_val_sample[i])

    # Original
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"True: {row.label} | Pred: {model.predict(X_val_sample[i:i+1], verbose=0)[0][0]:.2f}")
    axes[i, 0].axis('off')

    # Grad-CAM
    heatmap_cam = np.uint8(cm.jet(cam_heatmaps[i])[..., :3] * 255)
    axes[i, 1].imshow(img)
    axes[i, 1].imshow(heatmap_cam, cmap='jet', alpha=0.5)
    axes[i, 1].set_title("Grad-CAM")
    axes[i, 1].axis('off')

    # Grad-CAM++
    heatmap_plus = np.uint8(cm.jet(cam_plus_heatmaps[i])[..., :3] * 255)
    axes[i, 2].imshow(img)
    axes[i, 2].imshow(heatmap_plus, cmap='jet', alpha=0.5)
    axes[i, 2].set_title("Grad-CAM++")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig(f"{OUT}/Gradcam_tests.png", bbox_inches="tight")
plt.show()

## IG

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

def interpolate_images(baseline, image, alphas):
    """Generates interpolated images between baseline and original image."""
    alphas_x = alphas[:, tf.newaxis, tf.newaxis, tf.newaxis]
    baseline_x = tf.expand_dims(baseline, axis=0)
    input_x = tf.expand_dims(image, axis=0)
    delta = input_x - baseline_x
    return baseline_x + alphas_x * delta

def compute_gradients(images):
    """Computes gradients of the model's output probability w.r.t the images."""
    with tf.GradientTape() as tape:
        tape.watch(images)
        # We want to explain the probability of the positive class
        probs = model(images)[:, 0]
    return tape.gradient(probs, images)

def integrated_gradients(baseline, image, m_steps=50):
    """Approximates the integral of gradients along the path."""
    # 1. Generate alphas (interpolation steps)
    alphas = tf.linspace(start=0.0, stop=1.0, num=m_steps+1)

    # 2. Generate interpolated images
    interpolated_images = interpolate_images(baseline, image, alphas)

    # 3. Get gradients for all interpolated images
    path_gradients = compute_gradients(interpolated_images)

    # 4. Approximate the integral using the trapezoidal rule
    grads = (path_gradients[:-1] + path_gradients[1:]) / 2.0
    integrated_grads = tf.math.reduce_mean(grads, axis=0)

    # 5. Multiply by (image - baseline)
    return (image - baseline) * integrated_grads

# --- Visualization ---
# Use the X_val_sample from the previous Grad-CAM cell
fig, axes = plt.subplots(len(X_val_sample), 2, figsize=(8, 4 * len(X_val_sample)))

for i in range(len(X_val_sample)):
    img = tf.convert_to_tensor(X_val_sample[i], dtype=tf.float32)
    baseline = tf.zeros(shape=img.shape) # Black image as baseline

    # Calculate IG
    ig_attributions = integrated_gradients(baseline, img, m_steps=50)

    # Sum attributions across color channels for visualization
    ig_heatmap = tf.reduce_sum(tf.math.abs(ig_attributions), axis=-1)

    # Normalize for plotting
    ig_heatmap = (ig_heatmap - tf.reduce_min(ig_heatmap)) / (tf.reduce_max(ig_heatmap) - tf.reduce_min(ig_heatmap) + 1e-8)

    # Plot Original
    axes[i, 0].imshow(np.uint8(img.numpy()))
    axes[i, 0].set_title(f"Original Image {i+1}")
    axes[i, 0].axis('off')

    # Plot IG
    axes[i, 1].imshow(np.uint8(img.numpy()), alpha=0.5)
    axes[i, 1].imshow(ig_heatmap.numpy(), cmap='inferno', alpha=0.6)
    axes[i, 1].set_title("Integrated Gradients")
    axes[i, 1].axis('off')

plt.tight_layout()
plt.savefig(f"{OUT}/IG_tests.png", bbox_inches="tight")
plt.show()


##SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# 1. Create a background dataset (expected value reference)
# We take 50 random images from the training set to act as the "background"
background_images = []
for img, label in train_ds.unbatch().take(50):
    background_images.append(img.numpy())
background_images = np.array(background_images)

# 2. Initialize the GradientExplainer
# We pass the model and the background dataset
explainer = shap.GradientExplainer(model, background_images)

# 3. Compute SHAP values for our validation samples
# X_val_sample was defined in the Grad-CAM cell (shape: [N, 224, 224, 3])
# shap_values will represent the pixel-wise contribution to the prediction
print("Calculating SHAP values (this may take a minute)...")
shap_values = explainer.shap_values(X_val_sample)

# 4. Format for visualization
# For a single output (binary classification), shap_values is sometimes wrapped in a list
if isinstance(shap_values, list):
    shap_values = shap_values[0]

# 5. Plot using SHAP's built-in image plotter
# Red pixels increase the probability of it being a Myeloma cell.
# Blue pixels decrease the probability.
print("Red pixels = Push prediction towards Positive (Myeloma)")
print("Blue pixels = Push prediction towards Negative")

shap.image_plot(
    shap_values,
    np.uint8(X_val_sample),
    show=True
)

## Step 14: Look at the mistakes
**Why:** numbers say *how often* it's wrong; pictures hint at *why* (blurry crop, cell cut by the box, ambiguous cell).

In [ ]:
for title, sub in [("False positives (predicted positive, actually negative)", test[(test.pred == 1) & (test.label == 0)]),
                   ("False negatives (predicted negative, actually positive)", test[(test.pred == 0) & (test.label == 1)])]:
    sub = sub.sample(min(8, len(sub)), random_state=0) if len(sub) else sub
    fig, axes = plt.subplots(1, max(len(sub), 1), figsize=(2.6 * max(len(sub), 1), 3), squeeze=False)
    for ax in axes[0]: ax.axis("off")
    for ax, (_, r) in zip(axes[0], sub.iterrows()):
        ax.imshow(cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)); ax.set_title(f"{r.patient} p={r.prob:.2f}", fontsize=9)
    fig.suptitle(title); plt.show()

## Step 15: Save everything to Drive
**Why:** Colab's disk is wiped when the session ends. We keep the weights (rebuild with `build_vgg16()` then `model.load_weights(...)`), the metrics, the per-patient table and the exact split, so the experiment is reproducible.

In [ ]:
model.save_weights(f"{OUT}/vgg16.weights.h5")
json.dump(res, open(f"{OUT}/test_metrics.json", "w"), indent=2)
per.to_csv(f"{OUT}/per_patient_test.csv")
for n, d in [("train", tr), ("val", val), ("test", test)]:
    d[["path", "class_name", "patient", "photo", "label"]].to_csv(f"{OUT}/split_{n}.csv", index=False)
print("saved to", OUT)

## For the write-up
- Say exactly which folder was used (`detection/patients`, patients 01 to 10) and what the two classes are (Steps 3 and 5).
- Say the split was 60/10/30 **by patient** (give the patient lists and the actual cell percentages), and that the test set was used once. If you used `SPLIT_MODE = "photo"`, say patients were shared between splits.
- With 3 test patients and 1 validation patient the numbers are noisy. For a stronger claim, repeat with other patient splits (`MANUAL_SPLIT`) and report mean ± std.
- Report recall, not only accuracy, plus the per-patient table.
- Your design choices (ImageNet weights, two phases, what is unfrozen, augmentation, 224 px) are decisions to list.